<a href="https://colab.research.google.com/github/andres-merino/AprendizajeAutomaticoInicial-05-N0105/blob/main/2-Ejercicios/06-SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="border: none; border-collapse: collapse;">
    <tr>
        <td style="width: 20%; vertical-align: middle; padding-right: 10px;">
            <img src="https://i.imgur.com/nt7hloA.png" width="100">
        </td>
        <td style="width: 2px; text-align: center;">
            <font color="#0030A1" size="7">|</font><br>
            <font color="#0030A1" size="7">|</font>
        </td>
        <td>
            <p style="font-variant: small-caps;"><font color="#0030A1" size="5">
                <b>Facultad de Ciencias Exactas, Naturales y Ambientales</b>
            </font> </p>
            <p style="font-variant: small-caps;"><font color="#0030A1" size="4">
                Aprendizaje Automático Inicial &bull; Máquinas de soporte vectorial
            </font></p>
            <p style="font-style: oblique;"><font color="#0030A1" size="3">
                Andrés Merino &bull; Periodo 2026-2
            </font></p>
        </td>  
    </tr>
</table>

---
## <font color='264CC7'> Introducción </font>

A lo largo de este taller, aplicaremos el algoritmo de máquinas de soporte vectorial (SVM) para clasificar un conjunto de datos.

Los paquetes necesarios son:

In [ ]:
# Paquetes necesarios


---
## <font color='264CC7'> Clasificación </font>


### <font color='264CC7'> Preprocesamiento de datos </font>

Primero necesitas el conjunto de datos. Usa el conjunto de datos seleccionado en las primeras semanas.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio:</strong><br>
    Carga el conjunto de datos y procésalos:
<ul>
  <li>Muestra algunos datos.</li>
  <li>Muestra una descripción de los datos.</li>
  <li>Define predictores y objetivo, excluyendo información que revele la respuesta.</li>
  <li>Reserva prueba y divide el resto en entrenamiento y validación con semillas registradas; estratifica si se trata de clasificación y los tamaños de clase lo permiten.</li>
  <li>Ajusta imputación, codificación y escalado solo con entrenamiento; transforma validación sin reajustar.</li>
</ul>
</div>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Cargar el conjunto de datos
url = 'https://raw.githubusercontent.com/FJ039/Aprendizaje-Automatico/refs/heads/main/siniestros_transito_2019_limpio.csv'
df = pd.read_csv(url)

# Si existe una columna de índice innecesaria como 'Unnamed: 0', la eliminamos
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print("--- Primeras filas del conjunto de datos ---")
display(df.head())

print("\n--- Descripción de los datos ---")
display(df.describe(include='all'))

# 2. Definir predictores y objetivo
target_col = 'CLASE'
cols_to_drop = [target_col]

# Excluir columnas extras si las hubiera que revelen directamente la respuesta
X = df.drop(columns=cols_to_drop)
y = df[target_col]

# Identificar columnas numéricas y categóricas
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nPredictores numéricos: {num_cols}")
print(f"Predictores categóricos: {cat_cols}")
print(f"Variable objetivo: {target_col}")

# 3. Dividir en Entrenamiento, Validación y Prueba (Estratificado)
semilla = 42

# Primero separamos el 20% para prueba y el 80% restante (temporal)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=semilla, stratify=y
)

# Del 80% restante, separamos el 25% para validación (que equivale al 20% del total) y 75% para entrenamiento
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=semilla, stratify=y_temp
)

print(f"\nTamaño de Entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño de Validación: {X_val.shape[0]} muestras")
print(f"Tamaño de Prueba: {X_test.shape[0]} muestras")

# 4. Ajustar imputación, codificación y escalado solo con Entrenamiento y transformar
# Pipeline para datos numéricos: Imputación por la media + Escalado estándar
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline para datos categóricos: Imputación por el valor más frecuente + OneHot Encoding
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Transformador de columnas unificado
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Ajustar únicamente en entrenamiento y transformar
X_train_pre_array = preprocessor.fit_transform(X_train)

# Transformar validación y prueba sin reajustar
X_val_pre_array = preprocessor.transform(X_val)

X_test_pre_array = preprocessor.transform(X_test)

# Recuperar nombres de las columnas para reconstruir DataFrames
feature_names = preprocessor.get_feature_names_out()

X_train_pre = pd.DataFrame(X_train_pre_array, columns=feature_names)
X_val_pre = pd.DataFrame(X_val_pre_array, columns=feature_names)
X_test_pre = pd.DataFrame(X_test_pre_array, columns=feature_names)

print("\n--- Datos de entrenamiento procesados (primeras filas) ---")
display(X_train_pre.head())


### <font color='264CC7'> Determinación del mejor kernel </font>

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio:</strong><br>
    Determina el mejor kernel para el modelo SVM.
<ul>
  <li>Genera una gráfica que muestre la exactitud (accuracy) de entrenamiento y validación con diferentes kernels.</li>
  <li>Compara también C y, para RBF, gamma sobre las mismas particiones.</li>
  <li>Selecciona la configuración por validación, sin consultar prueba.</li>
</ul>
</div>



### <font color='264CC7'> Entrenamiento del modelo y evaluación </font>


<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio:</strong><br>
    Entrena el modelo y evalúalo:
<ul>
  <li>Fija kernel e hiperparámetros y reajusta las transformaciones y el modelo con entrenamiento y validación.</li>
  <li>Evalúa una sola vez en la prueba reservada, sin usarla para volver a ajustar.</li>
  <li>Genera una matriz de confusión y muestra las métricas de evaluación.</li>
</ul>
</div>

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio:</strong><br>
    Compara el modelo con otros algoritmos de clasificación vistos en clases anteriores usando las mismas particiones y métricas de validación; conserva prueba para evaluar una sola vez el modelo final seleccionado.
</div>